# US-04-04 — Hazelnut EXP-001 — Archive expérimentale

**Protocole non validé pour calibration ou comparaison scientifique.** La construction utilise les features sans pooling local ; le forward Anomalib 2.7.0 utilisé à l’inférence applique ce pooling. La restauration CPU/GPU concordante ne corrige pas cette différence. Les résultats et checkpoints EXP-001 sont conservés pour traçabilité, sans correction rétrospective.

Le code et les sorties ci-dessous sont ceux de l’export fourni ; seul cet avertissement Markdown a été ajouté. Voir [la fiche US-04-04](../docs/epic-04/us-04-04.md) avant toute réexécution. Le JSON de configuration Git décrit a posteriori cette expérience : il n’est pas lu par ces cellules.

Les cellules de restauration 1–5 ne contiennent aucune sortie dans cet export. Le rapport JSON séparé consigne PASS ; aucune réexécution locale n’est revendiquée. Pour rejouer : CPU, Internet activé, dataset v2 et Outputs GPU EXP-001 attachés. Le chargement sécurisé autorise uniquement TorchVersion après vérification du SHA-256.


In [ ]:

from pathlib import Path
import hashlib
import torch

EXPECTED_SHA256 = (
    "d8e6928416eee8172ae19d35630be99935ade2af7ae978ded50cbff2801faeec"
)

input_root = Path("/kaggle/input")

# Locate the checkpoint in the attached notebook outputs
candidates = [
    path
    for path in input_root.rglob("patchcore_model.pt")
    if path.parent.name == "EXP-001"
    and path.parent.parent.name == "hazelnut"
]

assert len(candidates) == 1, (
    f"Expected one Hazelnut checkpoint, found {len(candidates)}"
)

checkpoint_path = candidates[0]
artifact_dir = checkpoint_path.parent

required = [
    "patchcore_model.pt",
    "hazelnut_coreset.pt",
    "coreset_report.json",
    "inference_report.json",
    "reference_prediction.npz"
]

for filename in required:
    assert (artifact_dir / filename).is_file(), (
        f"Missing artifact: {filename}"
    )

# Verify checkpoint integrity
digest = hashlib.sha256()
with checkpoint_path.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        digest.update(chunk)

actual_sha256 = digest.hexdigest()

assert actual_sha256 == EXPECTED_SHA256, (
    "Checkpoint SHA256 mismatch"
)

# CPU-only verification
assert not torch.cuda.is_available(), (
    "Disable the GPU accelerator for this notebook"
)

print("=== HAZELNUT CPU RESTORE PREFLIGHT ===")
print("Device: CPU")
print("Torch:", torch.__version__)
print("Artifact directory:", artifact_dir)
print("Checkpoint size (MiB):",
      round(checkpoint_path.stat().st_size / 1024**2, 2))
print("SHA256:", actual_sha256)
print("Required artifacts:", len(required))

print("\nIndependent artifacts access: PASS")


In [ ]:

import sys
import subprocess
from importlib import metadata

required = {
    "anomalib": "2.7.0",
    "timm": "1.0.30",
    "lightning": "2.6.6",
}

def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

if any(
    installed_version(name) != version
    for name, version in required.items()
):
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "anomalib==2.7.0",
            "timm==1.0.30",
            "lightning==2.6.6",
        ],
        check=True
    )

import torch
import torchvision

assert torch.__version__ == "2.11.0+cpu"
assert torchvision.__version__ == "0.26.0+cpu"
assert not torch.cuda.is_available()

for package, expected in required.items():
    actual = installed_version(package)
    print(f"{package}: {actual}")
    assert actual == expected

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

print("\nCPU dependencies: PASS")


In [ ]:

import torch
from torch.torch_version import TorchVersion
from anomalib.models.image.patchcore.torch_model import PatchcoreModel

# The checkpoint SHA256 was already verified in preflight.
# Allow only the specific PyTorch type used in our metadata.
with torch.serialization.safe_globals([TorchVersion]):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

config = checkpoint["config"]
state_dict = checkpoint["state_dict"]

assert config["category"] == "hazelnut"
assert config["experiment_id"] == "EXP-001"
assert config["memory_bank_shape"] == [3205, 1536]

# Reconstruct architecture without downloading weights
restored_model = PatchcoreModel(
    backbone=config["backbone"],
    layers=config["layers"],
    pre_trained=False,
    num_neighbors=config["num_neighbors"]
).to("cpu")

# Restore backbone weights and memory bank
load_result = restored_model.load_state_dict(
    state_dict,
    strict=True
)

restored_model.feature_extractor.requires_grad_(False)
restored_model.eval()

assert not load_result.missing_keys
assert not load_result.unexpected_keys

assert tuple(restored_model.memory_bank.shape) == (
    3205, 1536
)
assert restored_model.memory_bank.device.type == "cpu"
assert torch.isfinite(restored_model.memory_bank).all()

assert not any(
    p.requires_grad
    for p in restored_model.feature_extractor.parameters()
)

print("=== HAZELNUT CPU RESTORATION ===")
print("Backbone:", config["backbone"])
print("Layers:", config["layers"])
print("Memory Bank:", tuple(restored_model.memory_bank.shape))
print("Device:", restored_model.memory_bank.device)
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

print("\nStrict CPU restoration: PASS")


In [ ]:

import sys
import subprocess
import csv
import hashlib
from pathlib import Path

REPO_DIR = Path("/kaggle/working/industrial-defect-ai")
SOURCE_COMMIT = "9c932580cbbe27c61d92905e344c8c7e86457381"

DATASET_ROOT = Path(
    "/kaggle/input/datasets/ipythonx/mvtec-ad"
)

MANIFEST_SHA256 = (
    "8f0e0d5b5bb767cd2d2e6f8a0ab10f65a959a97a38bb14bc5d9aae1d30c81fb6"
)

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        [
            "git", "clone", "--quiet",
            "https://github.com/MRdrissi/industrial-defect-ai.git",
            str(REPO_DIR)
        ],
        check=True
    )

subprocess.run(
    [
        "git", "-C", str(REPO_DIR),
        "checkout", "--detach", SOURCE_COMMIT
    ],
    check=True
)

actual_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True
).strip()

assert actual_commit == SOURCE_COMMIT

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.data.mvtec_dataset import MVTecDataset
from src.data.transforms import MVTecTransform

MANIFEST = (
    REPO_DIR / "docs/data/us-02-03/hazelnut_manifest.csv"
)

assert DATASET_ROOT.is_dir()
assert MANIFEST.is_file()

assert hashlib.sha256(
    MANIFEST.read_bytes()
).hexdigest() == MANIFEST_SHA256

transform = MVTecTransform(
    profile="patchcore",
    size=(256, 256)
)

val_dataset = MVTecDataset(
    dataset_root=DATASET_ROOT,
    manifest_path=MANIFEST,
    category="hazelnut",
    split="val",
    transform=transform
)

assert len(val_dataset) == 78

sample = val_dataset[0]

assert str(sample["image_path"]) == (
    "hazelnut/train/good/002.png"
)
assert int(sample["label"]) == 0

print("=== HAZELNUT CPU DATA PREPARATION ===")
print("Commit:", actual_commit)
print("Validation images:", len(val_dataset))
print("Reference:", sample["image_path"])
print("Input shape:", tuple(sample["image"].shape))

print("\nCPU data preparation: PASS")


In [ ]:

import json
import time
import hashlib
import numpy as np
import torch
from pathlib import Path

# ============================================
# US-04-04 / IDL-51
# Independent CPU inference verification
# ============================================

REFERENCE_FILE = (
    artifact_dir / "reference_prediction.npz"
)

RESTORE_OUTPUT = Path(
    "/kaggle/working/patchcore/hazelnut/EXP-001"
)
RESTORE_OUTPUT.mkdir(parents=True, exist_ok=True)

SCORE_TOL = 1e-3
MAP_TOL = 1e-3
RELATIVE_MAP_TOL = 1e-4

# 1. Read the saved GPU prediction
with np.load(REFERENCE_FILE, allow_pickle=False) as ref:
    gpu_score = np.asarray(
        ref["pred_score"], dtype=np.float32
    ).reshape(-1)

    gpu_map = np.asarray(
        ref["anomaly_map"], dtype=np.float32
    ).copy()

    reference_path = str(ref["image_path"].item())

assert reference_path == str(sample["image_path"])
assert gpu_score.shape == (1,)
assert gpu_map.shape == (1, 1, 256, 256)

# 2. Verify restored Memory Bank integrity
coreset_report = json.loads(
    (artifact_dir / "coreset_report.json").read_text()
)

restored_bank = (
    restored_model.memory_bank
    .detach()
    .cpu()
    .contiguous()
)

bank_hash = hashlib.sha256(
    restored_bank.numpy().tobytes()
).hexdigest()

assert bank_hash == coreset_report["memory_bank_sha256"]

# 3. Independent CPU prediction
restored_model.eval()

input_tensor = sample["image"].unsqueeze(0).to("cpu")

start = time.perf_counter()

with torch.inference_mode():
    prediction = restored_model(input_tensor)

cpu_duration = time.perf_counter() - start

cpu_score = (
    prediction.pred_score.detach()
    .cpu().numpy()
    .astype(np.float32)
    .reshape(-1)
)

cpu_map = (
    prediction.anomaly_map.detach()
    .cpu().numpy()
    .astype(np.float32)
)

assert cpu_score.shape == gpu_score.shape
assert cpu_map.shape == gpu_map.shape
assert np.isfinite(cpu_score).all()
assert np.isfinite(cpu_map).all()

# 4. Numerical comparison
score_diff = float(
    np.max(np.abs(cpu_score - gpu_score))
)

map_diff = np.abs(cpu_map - gpu_map)

map_max_diff = float(map_diff.max())
map_mean_diff = float(map_diff.mean())

map_range = float(
    gpu_map.max() - gpu_map.min()
)

map_relative_diff = (
    map_max_diff / map_range
    if map_range > 0 else map_max_diff
)

passed = (
    score_diff <= SCORE_TOL
    and map_max_diff <= MAP_TOL
    and map_relative_diff <= RELATIVE_MAP_TOL
)

# 5. Save independent restore report
report = {
    "category": "hazelnut",
    "experiment_id": "EXP-001",
    "device": "cpu",
    "reference_image": reference_path,
    "checkpoint_sha256": (
        "d8e6928416eee8172ae19d35630be99935ade2af7ae978ded50cbff2801faeec"
    ),
    "memory_bank_sha256": bank_hash,
    "gpu_score": float(gpu_score[0]),
    "cpu_score": float(cpu_score[0]),
    "score_abs_diff": score_diff,
    "map_max_abs_diff": map_max_diff,
    "map_mean_abs_diff": map_mean_diff,
    "map_relative_max_diff": map_relative_diff,
    "cpu_inference_seconds": cpu_duration,
    "tolerances": {
        "score_absolute": SCORE_TOL,
        "map_max_absolute": MAP_TOL,
        "map_relative_max": RELATIVE_MAP_TOL
    },
    "strict_load": True,
    "coreset_recomputed": False,
    "status": "PASS" if passed else "FAIL"
}

report_path = RESTORE_OUTPUT / "restore_report.json"

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

# 6. Results
print("=== HAZELNUT CPU / GPU COMPARISON ===")
print("Reference:", reference_path)
print("GPU score:", float(gpu_score[0]))
print("CPU score:", float(cpu_score[0]))
print("Score absolute difference:", score_diff)
print("Map maximum difference:", map_max_diff)
print("Map mean difference:", map_mean_diff)
print("Map relative difference:", map_relative_diff)
print("CPU inference duration (s):", round(cpu_duration, 3))
print("Memory Bank SHA256:", bank_hash)
print("Report:", report_path)

assert passed, "CPU/GPU numerical comparison failed"

print("\nINDEPENDENT HAZELNUT RESTORATION: PASS")


In [7]:

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

gpu_dir = Path(
    "/kaggle/input/notebooks/marwanedrissi/"
    "industrialdefect-ai-patchcore-hazelnut-memory-bank/"
    "patchcore/hazelnut/EXP-001"
)

cpu_dir = Path(
    "/kaggle/working/patchcore/hazelnut/EXP-001"
)

files = [
    (gpu_dir / "coreset_report.json", "coreset_report.json"),
    (gpu_dir / "inference_report.json", "inference_report.json"),
    (gpu_dir / "reference_prediction.npz", "reference_prediction.npz"),
    (cpu_dir / "restore_report.json", "restore_report.json"),
]

zip_path = Path(
    "/kaggle/working/us-04-04-evidence.zip"
)

with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:
    for source, name in files:
        assert source.is_file(), f"Missing: {source}"
        archive.write(source, arcname=name)

print("Evidence archive:", zip_path)
print("Size (KiB):", round(zip_path.stat().st_size / 1024, 2))

display(FileLink(str(zip_path)))


Evidence archive: /kaggle/working/us-04-04-evidence.zip
Size (KiB): 215.37


/kaggle/working/us-04-04-evidence.zip